# Unit V - Topic Modelling and Probabilistic Models
## Practical Implementation for BBA Students
---

In [ ]:
!pip install nltk scikit-learn pandas matplotlib seaborn gensim pyLDAvis wordcloud

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

## 1. Latent Dirichlet Allocation (LDA) — Topic Discovery

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sample documents (news articles)
documents = [
    # Sports articles
    "The football team won the championship match scoring three goals in the final game.",
    "Cricket world cup finals ended with a dramatic last ball finish between the teams.",
    "The tennis player won the grand slam tournament defeating the top ranked opponent.",
    "Basketball season started with exciting games and new player transfers.",
    "The athlete broke the world record in swimming at the Olympic games.",
    # Technology articles
    "Apple announced a new iPhone with advanced AI camera features and longer battery.",
    "Google released an update to its search algorithm improving artificial intelligence.",
    "The startup launched a cloud computing platform for small business digital transformation.",
    "New AI software can generate images and text using machine learning models.",
    "Cybersecurity threats increased as companies move to digital cloud platforms.",
    # Business articles
    "Stock market reached all time high as quarterly earnings exceeded analyst expectations.",
    "The central bank raised interest rates to control inflation in the economy.",
    "Global trade policies impacted export markets affecting small business revenue.",
    "The company reported strong financial growth and announced dividend payments to investors.",
    "Startup funding rounds attracted venture capital investment in emerging markets."
]

# Step 1: Convert to word counts (Bag of Words)
vectorizer = CountVectorizer(stop_words='english', max_features=100)
doc_term_matrix = vectorizer.fit_transform(documents)

print(f"Number of documents: {doc_term_matrix.shape[0]}")
print(f"Vocabulary size: {doc_term_matrix.shape[1]}")

# Step 2: Apply LDA with K=3 topics
lda_model = LatentDirichletAllocation(
    n_components=3,       # K = 3 topics
    random_state=42,
    max_iter=20
)
lda_model.fit(doc_term_matrix)

# Step 3: Show top words per topic (Beta - word-topic probabilities)
feature_names = vectorizer.get_feature_names_out()

print("\n=== DISCOVERED TOPICS (Top 8 words per topic) ===")
print("Beta (β) = probability that a word belongs to a topic\n")

topic_names = []
for topic_idx, topic in enumerate(lda_model.components_):
    top_word_indices = topic.argsort()[-8:][::-1]
    top_words = [feature_names[i] for i in top_word_indices]
    top_probs = [topic[i] / topic.sum() for i in top_word_indices]
    
    print(f"Topic {topic_idx + 1}:")
    for word, prob in zip(top_words, top_probs):
        bar = '█' * int(prob * 100)
        print(f"  {word:<15} β={prob:.3f} {bar}")
    print()

## 2. Per-Document Classification (Gamma)

In [ ]:
# Get document-topic distribution (Gamma)
doc_topic_dist = lda_model.transform(doc_term_matrix)

print("=== PER-DOCUMENT TOPIC DISTRIBUTION (Gamma γ) ===")
print("Each row shows what % of each topic a document contains\n")

topic_labels = ['Topic 1', 'Topic 2', 'Topic 3']

for i, doc in enumerate(documents):
    dominant_topic = np.argmax(doc_topic_dist[i]) + 1
    print(f"Doc {i+1}: \"{doc[:55]}...\"")
    for j, prob in enumerate(doc_topic_dist[i]):
        bar = '█' * int(prob * 20)
        marker = " ◄" if j == dominant_topic - 1 else ""
        print(f"  Topic {j+1}: {prob:.2%} {bar}{marker}")
    print()

# Visualize document-topic distribution as heatmap
import seaborn as sns

plt.figure(figsize=(10, 8))
doc_labels = [f'Doc {i+1}' for i in range(len(documents))]
sns.heatmap(doc_topic_dist, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=topic_labels, yticklabels=doc_labels)
plt.title('Document-Topic Distribution (Gamma)', fontsize=14, fontweight='bold')
plt.xlabel('Topics', fontsize=12)
plt.ylabel('Documents', fontsize=12)
plt.tight_layout()
plt.savefig('doc_topic_heatmap.png', dpi=100)
plt.show()

## 3. By-Word Topic Assignments

In [ ]:
# Show which topic each word most likely belongs to
print("=== BY-WORD TOPIC ASSIGNMENTS ===")
print("Which topic does each word most likely belong to?\n")

word_topic_assignments = {}
for word_idx, word in enumerate(feature_names):
    topic_probs = [lda_model.components_[t][word_idx] for t in range(3)]
    dominant = np.argmax(topic_probs)
    word_topic_assignments[word] = dominant + 1

# Group words by their dominant topic
for topic_id in range(1, 4):
    words_in_topic = [w for w, t in word_topic_assignments.items() if t == topic_id]
    print(f"Topic {topic_id} words: {', '.join(words_in_topic[:15])}")
    print()

## 4. Choosing the Right Number of Topics (K)

In [ ]:
# Test different values of K
k_values = range(2, 8)
perplexities = []

for k in k_values:
    lda = LatentDirichletAllocation(n_components=k, random_state=42, max_iter=20)
    lda.fit(doc_term_matrix)
    perplexity = lda.perplexity(doc_term_matrix)
    perplexities.append(perplexity)
    print(f"K={k}: Perplexity = {perplexity:.1f}")

# Plot perplexity curve
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), perplexities, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Topics (K)', fontsize=12)
plt.ylabel('Perplexity (lower is better)', fontsize=12)
plt.title('Choosing K: Perplexity vs Number of Topics', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('perplexity_curve.png', dpi=100)
plt.show()

print("\nHow to choose K:")
print("• Lower perplexity = better fit")
print("• Look for the 'elbow' where perplexity stops dropping sharply")
print("• Also check if topics make business sense!")

## 5. Topic Visualization with Word Clouds

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_list = ['Blues', 'Reds', 'Greens']

for topic_idx in range(3):
    # Get word weights for this topic
    topic = lda_model.components_[topic_idx]
    word_weights = {feature_names[i]: topic[i] for i in range(len(feature_names))}
    
    wc = WordCloud(width=400, height=300, background_color='white',
                   colormap=colors_list[topic_idx], max_words=20)
    wc.generate_from_frequencies(word_weights)
    
    axes[topic_idx].imshow(wc, interpolation='bilinear')
    axes[topic_idx].set_title(f'Topic {topic_idx + 1}', fontsize=16, fontweight='bold')
    axes[topic_idx].axis('off')

plt.suptitle('LDA Topic Word Clouds', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('topic_wordclouds.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Hidden Markov Model (HMM) — Simple POS Tagging

In [ ]:
import nltk
from nltk import word_tokenize, pos_tag
nltk.download('averaged_perceptron_tagger_eng')

# Simple HMM-style POS tagging demonstration
sentences = [
    "The cat sat on the mat.",
    "Apple released a new iPhone today.",
    "She runs fast in the morning.",
    "The stock market crashed yesterday."
]

# POS tag explanations for BBA students
tag_meanings = {
    'DT': 'Determiner (the, a, an)',
    'NN': 'Noun (cat, phone)',
    'NNP': 'Proper Noun (Apple, India)',
    'VBD': 'Verb Past (sat, released)',
    'VBZ': 'Verb Present (runs, is)',
    'IN': 'Preposition (on, in)',
    'JJ': 'Adjective (new, fast)',
    'RB': 'Adverb (today, yesterday)',
    'PRP': 'Pronoun (she, he)',
    'VB': 'Verb Base (run, go)'
}

print("=== HMM-STYLE POS TAGGING ===")
print("HMM assigns hidden states (POS tags) to observed words\n")

for sent in sentences:
    tokens = word_tokenize(sent)
    tagged = pos_tag(tokens)
    
    print(f"Sentence: \"{sent}\"")
    print(f"  {'Word':<15} {'POS Tag':<8} {'Meaning'}")
    print(f"  {'-'*45}")
    for word, tag in tagged:
        meaning = tag_meanings.get(tag, tag)
        print(f"  {word:<15} {tag:<8} {meaning}")
    print()

print("\n=== HMM CONCEPT ===")
print("Hidden states = POS tags (what we want to find)")
print("Observed      = The actual words (what we can see)")
print("Transition    = P(Verb after Noun) — patterns in tag sequences")
print("Emission      = P('cat' is a Noun) — word-tag associations")

## 7. Named Entity Recognition (CRF-style)

In [ ]:
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')
from nltk import ne_chunk

# CRF-style Named Entity Recognition
ner_sentences = [
    "Apple CEO Tim Cook announced new products in San Francisco.",
    "Google acquired DeepMind in London for 500 million dollars.",
    "Elon Musk founded SpaceX and Tesla in the United States.",
    "Microsoft partnered with OpenAI to develop artificial intelligence."
]

print("=== NAMED ENTITY RECOGNITION (NER) ===")
print("CRF looks at context from BOTH sides to identify entities\n")

for sent in ner_sentences:
    tokens = word_tokenize(sent)
    tagged = pos_tag(tokens)
    entities = ne_chunk(tagged)
    
    print(f"Sentence: \"{sent}\"")
    print("  Entities found:")
    for subtree in entities:
        if hasattr(subtree, 'label'):
            entity_name = ' '.join([word for word, tag in subtree])
            entity_type = subtree.label()
            print(f"    {entity_name} → {entity_type}")
    print()

print("Entity Types:")
print("  PERSON      = People's names")
print("  ORGANIZATION = Company/org names")
print("  GPE         = Geopolitical entities (countries, cities)")
print("  FACILITY    = Buildings, airports, etc.")

## 8. Complete Pipeline: End-to-End Topic Modelling

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Larger dataset of customer reviews
customer_reviews = [
    "The delivery was late and package was damaged. Terrible shipping experience.",
    "Product quality is amazing. Well made and durable material.",
    "Customer service was unhelpful. Had to wait hours on phone.",
    "Fast shipping and well packaged. Arrived in perfect condition.",
    "Poor quality material. Product broke after one week of use.",
    "Excellent customer support. They resolved my issue quickly.",
    "Shipping took two weeks. Product was not as described online.",
    "Great build quality. Sturdy and well designed product.",
    "The support team was responsive and helped exchange the item.",
    "Delivery was fast. But the product quality could be better.",
    "Worst customer service ever. No response to my emails.",
    "Beautiful product with excellent finish. Worth the price.",
    "Package arrived damaged. Delivery partner was careless.",
    "High quality materials and great attention to detail.",
    "Support chat was helpful. Got refund processed same day."
]

# Pipeline
print("=== COMPLETE TOPIC MODELLING PIPELINE ===")
print("\nStep 1: Text Preprocessing...")
vectorizer = CountVectorizer(stop_words='english', min_df=2)
dtm = vectorizer.fit_transform(customer_reviews)
print(f"  Created document-term matrix: {dtm.shape}")

print("\nStep 2: Running LDA with K=3 topics...")
lda = LatentDirichletAllocation(n_components=3, random_state=42, max_iter=30)
lda.fit(dtm)

print("\nStep 3: Interpreting Topics...")
words = vectorizer.get_feature_names_out()
suggested_names = ['', '', '']

for idx, topic in enumerate(lda.components_):
    top_words = [words[i] for i in topic.argsort()[-6:][::-1]]
    print(f"  Topic {idx+1}: {', '.join(top_words)}")

print("\nStep 4: Classifying reviews by dominant topic...")
doc_topics = lda.transform(dtm)
for i, review in enumerate(customer_reviews[:5]):
    dom = np.argmax(doc_topics[i]) + 1
    conf = np.max(doc_topics[i])
    print(f"  \"{review[:50]}...\" → Topic {dom} ({conf:.0%} confidence)")

print("\n=== BUSINESS ACTION ===")
print("• Topic about shipping → Send to Logistics team")
print("• Topic about quality → Send to Product team")
print("• Topic about service → Send to Customer Support team")